# Milestone 2: Parallel Document Embeddings (OpenMPI)

This notebook parallelizes the original NLP embedding pipeline using `mpi4py` over OpenMPI.

- Input: `Final Datasets/AlgoTesting_Merged_Coloums_Dataset.csv`
- Output: `Final Datasets/tfidf_vector.csv`, `Final Datasets/bge_embedding.csv`

To run with multiple ranks, launch the notebook kernel/process using OpenMPI (for example with `mpiexec -n 4 ...`). In a normal notebook run, it still works in single-rank mode.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import nltk
import spacy
from nltk.corpus import stopwords as nltk_sw
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from mpi4py import MPI

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()


def distribute_rows_round_robin(records, world_size):
    buckets = [[] for _ in range(world_size)]
    for row in records:
        buckets[row["idx"] % world_size].append(row)
    return buckets


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "Final Datasets").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing 'Final Datasets'.")


PROJECT_ROOT = find_project_root()
FINAL_DATASETS_DIR = PROJECT_ROOT / "Final Datasets"
INPUT_CSV = FINAL_DATASETS_DIR / "AlgoTesting_Merged_Coloums_Dataset.csv"
OUTPUT_TFIDF_CSV = FINAL_DATASETS_DIR / "tfidf_vector.csv"
OUTPUT_BGE_CSV = FINAL_DATASETS_DIR / "bge_embedding.csv"

if rank == 0:
    stage_times = {}
    pipeline_start_time = time.perf_counter()
    print(f"MPI world size: {size}")
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Input CSV: {INPUT_CSV}")
    print("[Stage 0/4] Initialization complete.")

In [ ]:
if rank == 0:
    stage_start = time.perf_counter()
    print("[Stage 1/4] Loading input data and distributing rows...")

    df = pd.read_csv(INPUT_CSV)
    source = df[["HADM_ID", "Combined_Data"]].dropna().reset_index(drop=True)
    records = [
        {
            "idx": i,
            "hadm_id": hadm_id,
            "text": str(combined_data),
        }
        for i, (hadm_id, combined_data) in enumerate(
            source.itertuples(index=False, name=None)
        )
    ]
    row_buckets = distribute_rows_round_robin(records, size)

    load_dist_elapsed = time.perf_counter() - stage_start
    stage_times["data_loading_distribution_seconds"] = load_dist_elapsed
    print(f"Loaded {len(records)} documents from input dataset.")
    print(f"[Stage 1/4] Completed in {load_dist_elapsed:.4f} seconds.")
else:
    row_buckets = None

local_records = comm.scatter(row_buckets, root=0)

if local_records:
    print(
        f"Rank {rank}: received {len(local_records)} rows "
        f"(idx {local_records[0]['idx']} -> {local_records[-1]['idx']})"
    )
else:
    print(f"Rank {rank}: received 0 rows.")

In [ ]:
preprocess_stage_start = time.perf_counter()

stopwords = set(nltk_sw.words("english"))
nlp = spacy.load("en_core_web_md")

print(f"Rank {rank}: preprocessing {len(local_records)} rows...")

local_processed = []
local_texts = [record["text"] for record in local_records]

for record, doc in zip(local_records, nlp.pipe(local_texts, batch_size=64)):
    clean_tokens = []
    raw_tokens = []
    for token in doc:
        lemma = str(token.lemma_)
        if lemma not in stopwords and lemma.isalpha():
            clean_tokens.append(lemma)
        raw_tokens.append(str(token))

    local_processed.append(
        {
            "idx": record["idx"],
            "hadm_id": record["hadm_id"],
            "clean_doc": " ".join(clean_tokens),
            "raw_doc": " ".join(raw_tokens),
        }
    )

processed_chunks = comm.gather(local_processed, root=0)
preprocess_stage_elapsed = time.perf_counter() - preprocess_stage_start
max_preprocess_elapsed = comm.allreduce(preprocess_stage_elapsed, op=MPI.MAX)

if rank == 0:
    all_processed = [item for chunk in processed_chunks for item in chunk]
    all_processed.sort(key=lambda x: x["idx"])

    idx_sequence = np.array([item["idx"] for item in all_processed], dtype=np.int64)
    expected_idx = np.arange(len(all_processed), dtype=np.int64)
    if not np.array_equal(idx_sequence, expected_idx):
        raise RuntimeError("Row-order integrity check failed during preprocessing gather.")

    hadm_ids = [item["hadm_id"] for item in all_processed]
    no_stopword_docs = [item["clean_doc"] for item in all_processed]
    raw_docs = [item["raw_doc"] for item in all_processed]

    stage_times["preprocessing_seconds"] = max_preprocess_elapsed
    print(f"Preprocessing complete for {len(all_processed)} documents (order preserved).")
    print(f"[Stage 2/4] Preprocessing completed in {max_preprocess_elapsed:.4f} seconds.")

In [ ]:
if rank == 0:
    print("[Stage 3/4] TF-IDF vectorization started...")
    tfidf_stage_start = time.perf_counter()

    count_vectorizer = CountVectorizer()
    tfidf_matrix = count_vectorizer.fit_transform(no_stopword_docs).toarray()

    tfidf_df = pd.DataFrame(tfidf_matrix, columns=count_vectorizer.get_feature_names_out())
    tfidf_df.insert(0, "HADM_ID", hadm_ids)
    tfidf_df.to_csv(OUTPUT_TFIDF_CSV, index=False)

    tfidf_stage_elapsed = time.perf_counter() - tfidf_stage_start
    stage_times["tfidf_vectorization_seconds"] = tfidf_stage_elapsed

    print(f"TF-IDF vectorization completed. Shape: {tfidf_matrix.shape}")
    print(f"Saved TF-IDF vectors to: {OUTPUT_TFIDF_CSV}")
    print(f"[Stage 3/4] Completed in {tfidf_stage_elapsed:.4f} seconds.")

In [ ]:
bge_stage_start = time.perf_counter()

local_indices = [item["idx"] for item in local_processed]
local_hadm_ids = [item["hadm_id"] for item in local_processed]
local_raw_docs = [item["raw_doc"] for item in local_processed]

print(f"Rank {rank}: encoding {len(local_raw_docs)} docs with BGE model.")

bge_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
if local_raw_docs:
    local_embeddings = bge_model.encode(
        local_raw_docs,
        show_progress_bar=False,
        batch_size=32,
        convert_to_numpy=True,
    )
else:
    emb_dim = bge_model.get_sentence_embedding_dimension()
    local_embeddings = np.empty((0, emb_dim), dtype=np.float32)

embedding_payload = {
    "idx": np.array(local_indices, dtype=np.int64),
    "hadm_id": np.array(local_hadm_ids, dtype=object),
    "emb": local_embeddings,
}

embedding_chunks = comm.gather(embedding_payload, root=0)
bge_stage_elapsed = time.perf_counter() - bge_stage_start
max_bge_elapsed = comm.allreduce(bge_stage_elapsed, op=MPI.MAX)

if rank == 0:
    idx_all = np.concatenate([chunk["idx"] for chunk in embedding_chunks])
    hadm_all = np.concatenate([chunk["hadm_id"] for chunk in embedding_chunks])
    emb_all = np.concatenate([chunk["emb"] for chunk in embedding_chunks], axis=0)

    order = np.argsort(idx_all)
    idx_sorted = idx_all[order]
    expected_idx = np.arange(idx_sorted.size, dtype=np.int64)
    if not np.array_equal(idx_sorted, expected_idx):
        raise RuntimeError("Row-order integrity check failed during embedding gather.")

    hadm_sorted = hadm_all[order]
    emb_sorted = emb_all[order]

    bge_df = pd.DataFrame(emb_sorted)
    bge_df.insert(0, "HADM_ID", hadm_sorted)
    bge_df.to_csv(OUTPUT_BGE_CSV, index=False)

    stage_times["bge_embedding_seconds"] = max_bge_elapsed

    print(f"BGE embedding shape: {emb_sorted.shape}")
    print(f"Saved BGE embeddings to: {OUTPUT_BGE_CSV} (order preserved)")
    print(f"[Stage 4/4] BGE embedding completed in {max_bge_elapsed:.4f} seconds.")

In [ ]:
comm.Barrier()
if rank == 0:
    total_pipeline_elapsed = time.perf_counter() - pipeline_start_time
    stage_times["total_pipeline_seconds"] = total_pipeline_elapsed

    print("Parallel document embedding pipeline finished successfully.")
    print("Generated files:")
    print(f"- {OUTPUT_TFIDF_CSV}")
    print(f"- {OUTPUT_BGE_CSV}")

    print("\nTiming summary (seconds):")
    for stage_name, stage_value in stage_times.items():
        print(f"  {stage_name}: {stage_value:.4f}")